# ARX 온도 모델 설명서 — 무엇을 재고, 결과가 어땠는가

`ARX_Temp_Model.py` 를 그대로 불러와서 실제 함수를 실행한 결과로 설명한다.

## 이 모델이 재는 것

탱크별로 **"지금부터 60초 동안 액추에이터(heat/exchanger/feeding)가 이만큼
명령을 받으면, 온도가 이만큼 변해야 정상이다"** 를 회귀식으로 학습한다.

```
ΔT(t → t+60s) = a·T(t) + Σ b·u_future + Σ c·u_past + d
```

그리고 이 식을 **feeding 을 뺀 모델 A** 와 **feeding 을 넣은 모델 B** 로 두 번
학습해서, feeding 이 온도로 설명되는 탱크와 안 되는 탱크를 가른다 — 이게 이
노트북에서 확인하는 핵심 결과다.

## 왜 이렇게 만들었는가 (요약)

- **예측 대상이 온도 수준이 아니라 변화량(ΔT)인 이유**: 수준을 맞히면 "현재값
  그대로 내놓기"만으로도 R² 가 아주 높게 나와서, 모델이 액추에이터를 실제로
  이해했는지 알 수 없다. 아래 1번에서 실측으로 보여준다.
- **입력에 미래 60초 액추에이터 상태를 넣는 이유**: 미래를 맞히려는 게 목적이
  아니라 "이만큼 명령이 나갔으면 온도가 이만큼 변해야 정상"을 모델링하는 게
  목적이다(릴레이 고장 감지용). 열 지연 때문에 과거 60초 항도 같이 넣는다.
- **heat 와 cool 을 동시에 못 넣는 이유**: 두 값이 완전 상보(둘 다 ON 인 순간이
  0건)라서 완전 다중공선성이 생긴다. heat 만 넣고, 그 계수를 "full-heat 와
  full-cool 의 차이"로 읽는다.
- **제외 구간**: 00:00~07:00(정체불명 운전 모드 의심), 알람(DIAGNOSTIC) 구간,
  라인 정지 구간. 윈도우가 제외 구간을 걸치는 행도 같이 버린다.

## 0. 모듈 로드

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

# 이 노트북은 parkkt/code_ongoing/ 에 있다. ARX_Temp_Model.py 도 같은 폴더에
# 있지만, Tank_Temp_Factor_Analysis.py 는 한 단계 위(parkkt/)에 있어서 경로를
# 추가해야 import 가 된다.
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))

import ARX_Temp_Model as ARX

# ARX_Temp_Model.py 는 __file__ 기준으로 SAVE_DIR/OUT_DIR 을 잡는데, 그 파일이
# parkkt/code_ongoing/ 로 옮겨진 뒤에도 실제 데이터(extracted_csv)와 이전 결과
# (analysis_out)는 여전히 한 단계 위 parkkt/ 에 있다. 그래서 두 경로만 보정한다.
PARENT = os.path.dirname(os.getcwd())
ARX.SAVE_DIR = os.path.join(PARENT, "extracted_csv")
ARX.OUT_DIR = os.path.join(PARENT, "analysis_out")

pd.set_option("display.width", 220)
DEMO_TANK = "I3"
print(f"데이터 폴더: {ARX.SAVE_DIR}  (존재={os.path.isdir(ARX.SAVE_DIR)})")

## 1. 왜 온도 수준이 아니라 변화량(ΔT)을 맞히는가

60초 뒤 온도(수준)를 직접 맞히려 하면, 아무 것도 안 하고 "지금 온도 그대로"
라고 찍기만 해도 R² 가 높게 나온다. 60초는 온도가 크게 안 바뀌는 시간이라서다.
이러면 모델이 액추에이터 효과를 배운 건지, 그냥 관성을 배운 건지 구분이 안 된다.

아래는 그 "현재값 그대로" 예측(naive persistence)의 R² 를 실제로 계산한 것이다.

In [ ]:
def naive_level_r2(tank_key):
    """수준을 '현재값 그대로'로 예측했을 때의 R² — 잔차가 곧 ΔT 자체다."""
    df, _ = ARX.build_frame(tank_key)
    T_future = df["T"] + df["y"]
    resid = df["y"].values  # T_future - T(t)
    r2 = 1 - np.sum(resid ** 2) / np.sum((T_future - T_future.mean()) ** 2)
    return float(r2), len(df)

for tk in ["I2", DEMO_TANK]:
    r2, n = naive_level_r2(tk)
    print(f"[{tk}] 수준을 '그대로 유지'로만 예측해도 R² = {r2:.3f}  (표본 {n:,}개)")
print("\n→ 그래서 이 모델은 수준이 아니라 ΔT(변화량) 자체를 목표로 삼는다.")

## 2. 특징 구성 — 미래/과거 60초 액추에이터 비율 + heat/cool 상보성

`build_frame` 이 만드는 특징:

- `T` — 현재 온도
- `y` — 목표: `T(t+60s) − T(t)`
- `{heat,exch,feed}_fut` — **미래** 60초 동안의 ON 비율 `[t, t+60)`
- `{heat,exch,feed}_past` — **과거** 60초 동안의 ON 비율 `[t-60, t)`

미래 항을 넣는 이유는 미래를 알아맞히려는 게 아니라, "이만큼 명령이 나갔다면
온도가 이만큼 반응해야 정상"이라는 기준선을 세우기 위해서다. 과거 항은 열
지연(명령이 나가도 온도 반응이 늦게 나타나는 것) 때문에 필요하다.

In [ ]:
demo_df, (both_on, both_off) = ARX.build_frame(DEMO_TANK)
print(f"[{DEMO_TANK}] 유효 표본 {len(demo_df):,}행")
demo_df[["T", "y", "heat_fut", "heat_past", "exch_fut", "exch_past", "feed_fut", "feed_past"]].head(6)

**heat/cool 상보성** — 둘 다 ON 이거나 둘 다 OFF 인 순간이 있는지 전체 탱크에서
확인한다. 0건이면 완전 다중공선성이라 cool 을 따로 못 넣는다(heat 만 넣고,
그 계수를 "완전 heat 상태와 완전 cool 상태의 차이"로 읽는다).

In [ ]:
rows = []
for tk in ARX.TANKS:
    _, (bon, boff) = ARX.build_frame(tk)
    rows.append({"탱크": tk, "heat/cool_둘다ON": bon, "heat/cool_둘다OFF": boff})
pd.DataFrame(rows)

모든 탱크에서 "둘 다 ON"은 0건이다 — 그래서 cool 은 뺀다. "둘 다 OFF"가 소수
존재하는 건 과도기 순간(전환 중) 정도로 무시할 수준이다.

**제외 구간**은 00:00~07:00, 알람(DIAGNOSTIC) 구간, 그리고 wagon 카운터가 최근
30분간 전혀 안 움직인 라인 정지 구간이다. 윈도우(±60초)가 이 구간에 조금이라도
걸치는 행도 같이 버린다(`ok` 마스크를 ±60초 롤링 최솟값으로 만드는 이유).

## 3. 모델 A vs 모델 B

| 모델 | 특징 |
|---|---|
| **A** | T, heat_fut/past, exch_fut/past |
| **B** | A + feed_fut/past |

시간순으로 앞 70%를 학습, 뒤 30%를 테스트로 쓴다(경계에 윈도우 겹침 방지용
구멍을 둔다). B 가 A 보다 **얼마나 더 잘 맞히는가**가 "그 탱크에서 feeding 이
온도로 검출 가능한가"에 대한 직접적인 증거다.

In [ ]:
A = ARX.fit_eval(demo_df, ARX.MODEL_A)
B = ARX.fit_eval(demo_df, ARX.MODEL_B)
print(f"[{DEMO_TANK}] 모델 A  R²={A['r2']:.3f}  RMSE={A['rmse']:.3f}  (학습 {A['n_train']:,} / 테스트 {A['n_test']:,})")
print(f"[{DEMO_TANK}] 모델 B  R²={B['r2']:.3f}  RMSE={B['rmse']:.3f}  ← feeding 을 넣으니 크게 개선")
print()
coef = pd.DataFrame({"특징": B["features"], "계수": B["beta"], "표준오차": B["se"]})
coef["t값"] = coef["계수"] / coef["표준오차"]
coef.round(4)

`feed_fut` 의 t값이 매우 크다(수십~수백) — 이 탱크에서는 feeding 이 온도
변화로 뚜렷하게 찍힌다는 뜻이다. 모든 탱크가 이렇지는 않다 — 아래 4번에서
전체를 비교한다.

## 4. 전체 탱크 비교 — feeding 이 온도로 보이는 탱크와 안 보이는 탱크

8개 탱크 전부에 대해 모델 A/B 를 학습해서 R²/RMSE 를 나란히 놓는다. 그리고
**feeding 이 실제로 켜져 있던 구간만 따로 골라** RMSE 를 다시 잰다 — feeding
은 전체 시간의 극히 일부(대략 1~수 %)라서, 전체 평균 RMSE 로는 그 구간의
개선이 묻혀 버리기 때문이다.

In [ ]:
summary, coef_rows = [], []
for tank in ARX.TANKS:
    df, (bon, boff) = ARX.build_frame(tank)
    if len(df) < 1000:
        print(f"⚠️ [{tank}] 유효 표본 {len(df)}개 — 건너뜀")
        continue

    a = ARX.fit_eval(df, ARX.MODEL_A)
    b = ARX.fit_eval(df, ARX.MODEL_B)

    mask = b["test_df"]["feed_fut"].values > 0
    if mask.sum() >= 30:
        fa = float(np.sqrt(np.mean((a["actual"][mask] - a["pred"][mask]) ** 2)))
        fb = float(np.sqrt(np.mean((b["actual"][mask] - b["pred"][mask]) ** 2)))
    else:
        fa = fb = np.nan

    summary.append({
        "탱크": tank, "학습": a["n_train"], "테스트": a["n_test"],
        "A_R2": round(a["r2"], 4), "B_R2": round(b["r2"], 4),
        "A_RMSE": round(a["rmse"], 3), "B_RMSE": round(b["rmse"], 3),
        "feed구간_n": int(mask.sum()),
        "feed구간_A_RMSE": round(fa, 3) if fa == fa else None,
        "feed구간_B_RMSE": round(fb, 3) if fb == fb else None,
    })
    for name, res in [("A", a), ("B", b)]:
        for f, coefv, se in zip(res["features"], res["beta"], res["se"]):
            coef_rows.append({"탱크": tank, "모델": name, "특징": f,
                              "계수": round(float(coefv), 4),
                              "t값": round(float(coefv / se), 2) if se > 0 else None})

summary_df = pd.DataFrame(summary)
coef_df = pd.DataFrame(coef_rows)
summary_df["R2_개선"] = (summary_df["B_R2"] - summary_df["A_R2"]).round(4)
summary_df.sort_values("R2_개선", ascending=False)

**결과 요약** (실행 시점 데이터 기준, 매일 데이터가 쌓이며 숫자는 조금씩
바뀐다):

- **I1 · I2 · I3 · P2 에서 feeding 을 넣었을 때 R² 가 뚜렷하게 오른다.**
  특히 I3(≈0.39→0.66), I1(≈0.14→0.49) 처럼 거의 두 배 넘게 뛴다.
  이 네 탱크는 이 노트북 위쪽 1번에서 확인한 대로 feeding 이 온도 변화로
  뚜렷하게 찍힌다는 뜻이다.
- **P1 · P3 · P4 · P5 는 R² 가 거의 그대로다.** feeding 계수 자체는 통계적으로
  유의(|t|>2)하게 나오는 경우도 있지만(P1, P4), 부호가 뒤섞이거나
  (feed_fut 는 음수, feed_past 는 양수) 전체 설명력에 주는 영향이 작다 —
  이미 exchanger/heat 항이 대부분의 변동을 설명하고 있어서 feeding 이 덧붙일
  여지가 적기 때문으로 보인다.
- 이 결과는 `Feeding_Model_설명 copy.ipynb` 에서 **수위** 기준으로 이미 확인한
  관찰("온도로는 8개 중 4개(I1/I2/I3/P2)만 feeding 효과가 검출된다")과
  **정확히 같은 네 탱크**로 일치한다 — 서로 다른 두 방법(온도 ARX vs 수위
  Cohen's d)이 같은 결론에 도달했다는 교차검증이다.

In [ ]:
feed_coef = coef_df[(coef_df["모델"] == "B") & (coef_df["특징"].isin(["feed_fut", "feed_past"]))]
feed_coef.pivot(index="탱크", columns="특징", values=["계수", "t값"])

## 5. 시각화 — 테스트 구간 실제 vs 예측 ΔT

`analysis_out/ARX_<탱크>_pred.png` 로 저장해 둔 그림과 같은 방식으로, 데모
탱크 하나를 노트북 안에서 바로 그려서 확인한다.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(B["test_index"], B["actual"], color="#444444", linewidth=0.8, label="실제 ΔT")
ax.plot(B["test_index"], B["pred"], color="#d62728", linewidth=0.8, alpha=0.8, label="예측 ΔT (모델 B)")
ax.set_title(f"{DEMO_TANK} — 60초 온도변화 예측 (테스트 구간)  R²={B['r2']:.3f}  RMSE={B['rmse']:.2f}")
ax.set_ylabel("ΔT (60초)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

## 6. 한계

- **탱크마다 feeding 이 온도로 검출되는 정도가 다르다.** P1/P3/P4/P5 처럼
  exchanger/heat 신호가 온도 변화를 이미 대부분 설명하는 탱크에서는 feeding
  계수가 유의해도 실질적인 예측력 개선은 작다 — 이런 탱크는 `Feeding_Model.py`
  처럼 **수위**를 직접 보는 방법이 더 적합하다.
- **heat/cool 은 계수를 따로 못 가른다.** 완전 상보라서 heat 계수는 항상
  "heat 상태와 cool 상태의 차이"로만 읽어야 한다.
- **원인은 구분 못 한다.** "명령 대비 온도 반응이 정상 범위인가"까지만 말한다.
- **00~07시·라인 정지 구간은 학습/평가 대상이 아니다.** 그 구간에 적용하면
  안 된다.